# Reference-data generation (pyscf / gpu4pyscf)

Folder-staged pipeline: each stage cell drains its input folder into the next, so a
molecule advances simply by running cells top-to-bottom. Stages are **idempotent and
resumable** — re-running a cell skips items already produced, so a Colab reconnect just
means re-running the cells.

```
00_inbox  ->  01_optimized  ->  02_frequencies  ->  03_wigner  ->  04_labeled
```

Per molecule: optimize (wB97M-V/def2-svpd) → harmonic frequencies → Wigner sampling
(300 K, 500 structures) → label each with energy/forces/dipole/quadrupole/
polarizability/dipole-derivatives → extended-XYZ matching `data/labels/*.extxyz`.

The compute core prefers **gpu4pyscf** (Colab GPU) and falls back to **pyscf** (local CPU).

## 1. Install
On Colab (GPU) this installs the gpu4pyscf stack; locally it installs the CPU pyscf backend.
Either way it then installs the `rsfff` package in editable mode.

In [16]:
import sys, subprocess, os, shutil, importlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # GPU stack (mirrors notebooks/almo_eda_pyscf.ipynb) + geomeTRIC for opt.
    # Analytic CPSCF response comes from gpu4pyscf.properties on this path.
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "pyscf==2.8.0", "basis-set-exchange==0.11", "geometric==1.1.0",
        "cupy-cuda12x==13.4.1", "cutensor-cu12==2.2.0",
        "gpu4pyscf-cuda12x==1.7.6"], check=True)

    # Do NOT name the clone directory "rsfff". Colab puts /content on sys.path, so
    # a bare ./rsfff directory becomes an implicit *namespace package* that merges
    # with the installed one: submodules still import, but rsfff.__file__ is None.
    # Cloning to a non-colliding name keeps rsfff a normal package.
    if os.path.isdir(os.path.join("rsfff", ".git")):
        shutil.rmtree("rsfff")
        print("removed ./rsfff clone (its name shadowed the rsfff package)")

    REPO_DIR, REPO_URL = "rsfff_repo", "https://github.com/heindelj/rsfff.git"
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
    print("rsfff repo at commit:", head)

    # --no-deps is important: rsfff's full dependency list includes torch-cluster,
    # which has no Colab wheel and compiles from source (20-40+ min). rsfff.qcgen
    # needs only numpy + pyscf, both already installed above, so we install the
    # package itself and skip dependency resolution entirely.
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "--no-deps"], check=True)
    REPO_PATH = Path(REPO_DIR).resolve()
else:
    # Local CPU backend. VV10 (needed by wB97M-V) is built into pyscf; we do NOT
    # install pyscf-dispersion (D3/D4 only, and its release is broken vs new numpy).
    # pyscf-properties supplies the analytic CPHF polarizability on CPU; joblib
    # parallelizes the label stage (the macOS pyscf wheel has no OpenMP, so a
    # single SCF is stuck on one core no matter what OMP_NUM_THREADS says).
    subprocess.run([sys.executable, "-m", "pip", "install", "pyscf>=2.14",
                    "geometric", "pyscf-properties", "joblib>=1.3"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".."], check=True)
    REPO_PATH = Path("..").resolve()

# Drop any stale/shadowed cached import of rsfff so the next cell picks up the
# version we just installed (no kernel restart needed).
for _m in [k for k in list(sys.modules) if k == "rsfff" or k.startswith("rsfff.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# REPO_PATH is the single source of truth for repo-relative data (see the seed
# cell); it never depends on rsfff.__file__, which is None under a namespace
# package, nor on the notebook's CWD.
assert (REPO_PATH / "data" / "mols").is_dir(), f"no data/mols under {REPO_PATH}"
print("install cell done; IN_COLAB =", IN_COLAB)
print("repo :", REPO_PATH)

Obtaining file:///Users/joseph.heindel/dev/rsfff
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for rsfff (pyproject.toml): started
  Building editable for rsfff (pyproject.toml): finished with status 'done'
  Created wheel for rsfff: filename=rsfff-0.1.0-0.editable-py3-none-any.whl size=3088 sha256=03352ab4c53ee637f9b13485176a958b2e373e994ce87202fccbb0c981fc0342
  Stored in directory: /private/var/folders/g1/f38hz3r94jl6s36kkjb54z100000gp/T/pip-ephem-wheel-cache-mqugfp0l/wheels/dd/3a/38/58a7b35bf1a97a4f03726fffb87b308b45873e

## 2. Setup
Pick the pipeline root (Colab Drive folder or a local directory), create the stage folders,
and report which backend is active.

In [17]:
import os
# macOS libomp guard (harmless elsewhere); must be set before importing pyscf.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path
from rsfff.qcgen import pipeline as pl
from rsfff.qcgen.backend import backend_name

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/QC_runs")
else:
    # Local: keep runs under the repo's data/ tree.
    ROOT = Path("..") / "data" / "qc_runs"

cfg = pl.PipelineConfig(
    root=ROOT,
    xc="wB97M-V",
    basis="def2-svpd",
    temperature=300.0,
    n_samples=500,
    seed=0,
    # Analytic coupled-perturbed SCF for the polarizability (3 linear solves
    # instead of 6 SCF runs). "finite-difference" for an independent cross-check.
    response="cpscf",
    # Dipole derivatives (atomic polar tensors) are OFF: they dominated labeling
    # cost (a Hessian object + 3N coupled-perturbed solves, vs 3 for everything
    # else). With them off the dipole_derivatives key is absent from the extxyz
    # header. Flip to True if you want them back.
    with_dipole_derivatives=False,
    # Worker processes for the label stage (structures are independent). Held at
    # 10 rather than every core, to leave headroom for other work on the machine.
    # Ignored on the GPU backend, which is pinned to 1 (a single device).
    n_workers=10,
)
pl.make_dirs(cfg)
print("backend :", backend_name())
print("root    :", cfg.root.resolve())
print("method  :", cfg.xc, "/", cfg.basis, "| T =", cfg.temperature, "K | N =", cfg.n_samples)
print("response:", cfg.response, "| dipole derivatives:", cfg.with_dipole_derivatives)
print("workers :", pl.resolve_workers(cfg))

backend : pyscf
root    : /Users/joseph.heindel/dev/rsfff/data/qc_runs
method  : wB97M-V / def2-svpd | T = 300.0 K | N = 500
response: cpscf | dipole derivatives: False
workers : 10


## 3. Seed the inbox
Copy the molecules you want to process into `00_inbox/`. Charges/multiplicities are looked up
in `pipeline.SPECIES` (open-shell allowed, e.g. `oh_radical` → doublet). Edit `NAMES` to choose
which species to run; `None` seeds everything in `data/mols/`.

In [18]:
# REPO_PATH comes from the install cell and works on Colab and locally alike.
# (Deliberately not derived from rsfff.__file__, which is None when a same-named
# directory on sys.path turns rsfff into a namespace package.)
MOLS = REPO_PATH / "data" / "mols"

NAMES = None  # subset to run; None = every *.xyz in data/mols

if NAMES is None:
    xyz_paths = sorted(MOLS.glob("*.xyz"))
else:
    xyz_paths = [MOLS / f"{n}.xyz" for n in NAMES]

missing = [p.name for p in xyz_paths if not p.exists()]
assert not missing, f"missing geometries in {MOLS}: {missing}"

seeded = pl.seed_inbox(cfg, xyz_paths)
print(f"seeded {len(seeded)} file(s) into {cfg.root / pl.INBOX}:")
print(" ", ", ".join(seeded) if seeded else "(nothing new; inbox already populated)")
# You can also drop your own .xyz files directly into the inbox folder above and
# the stages will pick them up. On Colab that folder lives on Google Drive
# (ROOT = /content/drive/MyDrive/QC_runs), so inputs and outputs both persist
# across sessions -- no separate Drive plumbing needed.

seeded 12 file(s) into ../data/qc_runs/00_inbox:
  ch4.xyz, co2.xyz, h2o.xyz, h2so4.xyz, h3o+.xyz, h_atom.xyz, hf.xyz, hso4-.xyz, nh3.xyz, oh-.xyz, oh_radical.xyz, so42-.xyz


## 4. Stage: optimize geometries
`00_inbox/*.xyz` → `01_optimized/<name>.xyz` (+ `<name>.json` meta).

In [19]:
pl.run_stage(pl.STAGE_OPTIMIZE, cfg)

[18:20:50] stage 'optimize' [pyscf]: 12 candidate(s) in 00_inbox/
[18:20:50]   optimize: ch4 ...


geometric-optimize called with the following command line:
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/ipykernel_launcher.py --f=/Users/joseph.heindel/Library/Jupyter/runtime/kernel-v3280333d0dafbd2160471decee3b69847585b4e0f.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%

[18:22:30]   optimize: ch4 done (99.7s)
[18:22:30]   optimize: co2 ...


geometric-optimize called with the following command line:
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/ipykernel_launcher.py --f=/Users/joseph.heindel/Library/Jupyter/runtime/kernel-v3280333d0dafbd2160471decee3b69847585b4e0f.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%

[18:23:45]   optimize: co2 done (75.1s)
[18:23:45]   optimize: h2so4 ...


...

geometric-optimize called with the following command line:
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/ipykernel_launcher.py --f=/Users/joseph.heindel/Library/Jupyter/runtime/kernel-v3280333d0dafbd2160471decee3b69847585b4e0f.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%

[18:51:53]   optimize: oh_radical done (23.5s)
[18:51:53]   optimize: so42- ...


geometric-optimize called with the following command line:
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/ipykernel_launcher.py --f=/Users/joseph.heindel/Library/Jupyter/runtime/kernel-v3280333d0dafbd2160471decee3b69847585b4e0f.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%

[18:58:50]   optimize: so42- done (417.4s)
[18:58:50] stage 'optimize': 10 done, 1 skipped, 1 failed


(['ch4',
  'co2',
  'h2so4',
  'h3o+',
  'hf',
  'hso4-',
  'nh3',
  'oh-',
  'oh_radical',
  'so42-'],
 ['h2o'],
 ['h_atom'])

## 5. Stage: harmonic frequencies
`01_optimized/` → `02_frequencies/<name>.npz` (Hessian, masses, geometry). Analytic Hessian
where supported; automatic finite-difference fallback for UKS + NLC (open-shell wB97M-V).

In [20]:
pl.run_stage(pl.STAGE_FREQUENCIES, cfg)

[18:58:51] stage 'frequencies' [pyscf]: 11 candidate(s) in 01_optimized/
[18:58:51]   frequencies: ch4 ...
[19:06:47]   frequencies: ch4 done (476.1s)
[19:06:47]   frequencies: co2 ...
[19:11:07]   frequencies: co2 done (260.3s)
[19:11:07]   frequencies: h2so4 ...
[20:05:44]   frequencies: h2so4 done (3276.7s)
[20:05:44]   frequencies: h3o+ ...
[20:10:31]   frequencies: h3o+ done (287.8s)
[20:10:31]   frequencies: hf ...
[20:11:23]   frequencies: hf done (51.3s)
[20:11:23]   frequencies: hso4- ...
[20:47:57]   frequencies: hso4- done (2194.0s)
[20:47:57]   frequencies: nh3 ...
[20:52:25]   frequencies: nh3 done (268.1s)
[20:52:25]   frequencies: oh- ...
[20:53:21]   frequencies: oh- done (56.0s)
[20:53:21]   frequencies: oh_radical ...
[20:54:29]   frequencies: oh_radical done (68.7s)
[20:54:29]   frequencies: so42- ...
[21:18:49]   frequencies: so42- done (1459.4s)
[21:18:49] stage 'frequencies': 10 done, 1 skipped, 0 failed


(['ch4',
  'co2',
  'h2so4',
  'h3o+',
  'hf',
  'hso4-',
  'nh3',
  'oh-',
  'oh_radical',
  'so42-'],
 ['h2o'],
 [])

## 6. Stage: Wigner sampling
`02_frequencies/` → `03_wigner/<name>.extxyz` (`n_samples` geometries at `temperature`).

In [21]:
pl.run_stage(pl.STAGE_WIGNER, cfg)

[21:18:49] stage 'wigner' [pyscf]: 11 candidate(s) in 02_frequencies/
[21:18:49]   wigner: ch4 ...
[21:18:49]   wigner: ch4 done (0.0s)
[21:18:49]   wigner: co2 ...
[21:18:49]   wigner: co2 done (0.0s)
[21:18:49]   wigner: h2so4 ...
[21:18:49]   wigner: h2so4 done (0.0s)
[21:18:49]   wigner: h3o+ ...
[21:18:49]   wigner: h3o+ done (0.0s)
[21:18:49]   wigner: hf ...
[21:18:49]   wigner: hf done (0.0s)
[21:18:49]   wigner: hso4- ...
[21:18:49]   wigner: hso4- done (0.0s)
[21:18:49]   wigner: nh3 ...
[21:18:49]   wigner: nh3 done (0.0s)
[21:18:49]   wigner: oh- ...
[21:18:49]   wigner: oh- done (0.0s)
[21:18:49]   wigner: oh_radical ...
[21:18:49]   wigner: oh_radical done (0.0s)
[21:18:49]   wigner: so42- ...
[21:18:49]   wigner: so42- done (0.0s)
[21:18:49] stage 'wigner': 10 done, 1 skipped, 0 failed


(['ch4',
  'co2',
  'h2so4',
  'h3o+',
  'hf',
  'hso4-',
  'nh3',
  'oh-',
  'oh_radical',
  'so42-'],
 ['h2o'],
 [])

## 7. Stage: label properties
`03_wigner/` → `04_labeled/<name>.extxyz` (energy, forces, dipole, quadrupole,
polarizability, dipole derivatives). On success the original inbox `.xyz` is archived to
`_completed/`. This is the expensive stage; it streams frames to disk and writes atomically.

In [ ]:
pl.run_stage(pl.STAGE_LABEL, cfg)

[21:18:49] stage 'label' [pyscf]: 11 candidate(s) in 03_wigner/
[21:18:49]   label: ch4 ...
[21:18:49]     ch4: labeling 500 frames on 10 worker(s)


/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/rhf.py:45: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/dhf.py:30: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/rhf.py:45: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/dhf.py:30: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/zfs/uhf.py:40: UserWarning: Module ZFS is under testing
  warnings.warn('Module ZFS is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/zfs/uhf.py:40: UserWarning: Module ZFS is under testing
  warnings.warn('Module ZFS

[21:30:13]     ch4: labeled 25/500
[21:37:44]     ch4: labeled 50/500
[21:48:48]     ch4: labeled 75/500
[21:56:27]     ch4: labeled 100/500
[22:07:16]     ch4: labeled 125/500
[22:14:52]     ch4: labeled 150/500
[22:25:26]     ch4: labeled 175/500
[22:33:33]     ch4: labeled 200/500
[22:43:40]     ch4: labeled 225/500
[22:51:46]     ch4: labeled 250/500
[23:01:55]     ch4: labeled 275/500
[23:10:12]     ch4: labeled 300/500
[23:20:36]     ch4: labeled 325/500
[23:28:47]     ch4: labeled 350/500
[23:39:00]     ch4: labeled 375/500
[23:46:58]     ch4: labeled 400/500
[23:57:23]     ch4: labeled 425/500
[00:05:19]     ch4: labeled 450/500
[00:15:31]     ch4: labeled 475/500
[00:23:19]     ch4: labeled 500/500
[00:23:19]     ch4: 500 labeled, 0 skipped
[00:23:19]   label: ch4 done (11069.8s)
[00:23:19]   label: co2 ...
[00:23:19]     co2: labeling 500 frames on 10 worker(s)
[00:31:35]     co2: labeled 25/500
[00:37:11]     co2: labeled 50/500
[00:45:31]     co2: labeled 75/500
[00:51:12] 

## 8. Collect + summary
Concatenate all labeled species into one dataset file and report per-species frame counts and
any failures (tracebacks live in `_failed/`).

In [ ]:
out_path, counts = pl.collect(cfg)
print("combined dataset ->", out_path.resolve())
for name, n in sorted(counts.items()):
    print(f"  {name:14s} {n} frames")

failed = sorted((cfg.root / pl.FAILED).glob("*.error.log"))
if failed:
    print("\nfailures:")
    for f in failed:
        print("  ", f.name)

combined dataset -> /Users/joseph.heindel/dev/rsfff/data/qc_runs/reference_dataset.extxyz
  h2o            500 frames
